# Data Preprocessing
This notebook focuses on loading the raw data and applying preprocessing steps to clean, transform, and prepare the dataset for further analysis and modeling.

In [ ]:
import pandas as pd
import numpy as np
import os
from pathlib import Path
import sys

project_root = Path.cwd().parent  
sys.path.append(str(project_root))

In [ ]:
# Load raw datasets
market_price = pd.read_csv("../data/raw/Zeitreihen_Marktpreis_20180930_20260613_UTC.csv", sep=";")
rebap = pd.read_csv("../data/raw/reBAP unterdeckt [2026-06-13 09-51-13].csv", sep=";")
wka = pd.read_excel("../data/raw/Zeitreihen_WKA_2025_2026.xlsx")
weather = pd.read_excel("../data/raw/Zeitreihen_Wetterprognosen_2025_2026.xlsx")

## 1. Explore Missing Values & Duplicates
First, let's identify any data quality issues like missing data or duplicate rows.

In [ ]:
datasets = {
    'Market Price': market_price,
    'reBAP': rebap,
    'WKA': wka,
    'Weather': weather
}

for name, df in datasets.items():
    print(f"--- {name} ---")
    print(f"Missing values:\n{df.isnull().sum().sum()} total missing cells")
    if df.isnull().sum().sum() > 0:
        print(df.isnull().sum()[df.isnull().sum() > 0])
    print(f"Duplicate rows: {df.duplicated().sum()}\n")

## 2. Handle Missing Values & Duplicates
Here you can choose the approach to handle missing values and duplicates. Common approaches:
- `drop_duplicates`: remove exact duplicate rows
- `dropna`: completely remove rows with missing values
- `fillna`: fill missing values (e.g., with 'ffill', 'bfill', mean, median, or a specific value)
- `interpolate`: interpolate missing values

In [ ]:
# Choose the method to handle missing values
METHOD = 'ffill' # Options: 'dropna', 'ffill', 'bfill', 'interpolate', 'mean'

def clean_dataframe(df, method):
    # Remove duplicates
    df = df.drop_duplicates()
    
    # Handle missing values based on selected method
    if method == 'dropna':
        df = df.dropna()
    elif method == 'ffill':
        df = df.fillna(method='ffill')
    elif method == 'bfill':
        df = df.fillna(method='bfill')
    elif method == 'interpolate':
        df = df.interpolate(method='linear')
    elif method == 'mean':
        df = df.fillna(df.mean(numeric_only=True))
    return df

market_price = clean_dataframe(market_price, METHOD)
rebap = clean_dataframe(rebap, METHOD)
wka = clean_dataframe(wka, METHOD)
weather = clean_dataframe(weather, METHOD)

print("Data cleaning applied successfully using method:", METHOD)

## 3. Save Preprocessed Data

In [ ]:
# Save data to the preprocessed folder
os.makedirs("../data/preprocessed", exist_ok=True)

market_price.to_csv("../data/preprocessed/market_price_clean.csv", index=False)
# rebap.to_csv("../data/preprocessed/rebap_clean.csv", index=False)
# wka.to_csv("../data/preprocessed/wka_clean.csv", index=False)
# weather.to_csv("../data/preprocessed/weather_clean.csv", index=False)